In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
sys.path.append("..") 
# sys.path.append("C:/Users/ngoak/OneDrive/Projects/agentic")

from typing import List

import numpy as np
import cv2
import matplotlib.pyplot as plt
import PIL

import torch
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.nn.functional as F

from transformers import CLIPSegConfig, CLIPSegProcessor
from transformers import CLIPSegForImageSegmentation

from configuration_refiner import RefinerConfig
from modeling_refiner import Refiner, RefinerDecoderOutput

from kitti_tracking import KittiDataset

from segmentation import SemanticSegmentationTool
from inpainting import InpaintingTool

In [3]:
def _collate_fn(data):
    image_list, label_list = zip(*data)
    return image_list, label_list

root_dir = "E:/KittiTracking"
n_steps, n_pred_steps = 8, 3

train_ds = KittiDataset(root_dir, "train", n_steps, n_pred_steps)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=_collate_fn)

In [ ]:
clip_segm_name = "CIDAS/clipseg-rd64-refined"
clip_segm_processor = CLIPSegProcessor.from_pretrained(clip_segm_name, use_fast=True)
clip_segm_model = CLIPSegForImageSegmentation.from_pretrained(clip_segm_name).to("cuda:0")
for p in clip_segm_model.parameters(): p.requires_grad_(False)

def sem_segm_tool(frame: PIL.Image.Image, prompt: List[str]) -> PIL.Image.Image:
	inputs = clip_segm_processor(
		images=[frame]*len(prompt),
		text=prompt,
		padding=True,
		truncation=True,
		return_tensors="pt"
	).to("cuda")
	outputs = clip_segm_model(**inputs)
	
	# (K, H, W)
	rand = torch.rand(len(prompt)).to("cuda")
	rand = rand > 0
	rand = rand.unsqueeze(-1).unsqueeze(-1)
	mask = (outputs.logits*rand).amax(0).sigmoid().float().cpu().numpy()
	
	mask = PIL.Image.fromarray((mask > 0.5).astype(np.uint8) * 255).resize((352, 352))
	return mask.resize(frame.size)


# sem_mask_list_flatten = [sem_segm_tool(frame, [prompt]) for frame in frame_list_flatten]

In [5]:
# models
# sem_segm_tool = SemanticSegmentationTool(model_name="CIDAS/clipseg-rd64-refined")
inpainting_tool = InpaintingTool(ckpt='../../../../checkpoints/dstt.pth')

model_name = "CIDAS/clipseg-rd64-refined"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

clipseg_config = CLIPSegConfig.from_pretrained(model_name)
refiner_config = RefinerConfig(action_channels=2, **clipseg_config.to_dict())
refiner_processor = CLIPSegProcessor.from_pretrained(model_name, use_fast=True)
refiner = Refiner._from_config(refiner_config).to(device)
for p in refiner.clip.parameters(): p.requires_grad = False

In [14]:
def create_optimizer(model, lr=5e-5, weight_decay=0.01):

    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]

    optimizer_grouped_parameters = [
        {
            "params": [
                p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": weight_decay,
        },
        {
            "params": [
                p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]

    optimizer = torch.optim.AdamW(
        optimizer_grouped_parameters,
        lr=lr,
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    return optimizer

prompt = "car"

optimizer = create_optimizer(refiner.decoder, lr=3e-5)

for epoch in range(10):
	for batch in train_loader:
		frame_list, label_list = batch
		b = len(frame_list)
		ws, hw = zip(*[frames[0].size for frames in frame_list])

		frame_list_flatten = [frame for frames in frame_list for frame in frames[:n_steps]]
		sem_mask_list_flatten = [sem_segm_tool(frame, [prompt]) for frame in frame_list_flatten]
		# masked_frame_list_flatten = [
		# 	PIL.Image.composite(frame, PIL.Image.new("RGB", frame.size), mask)
		# 	for frame, mask in zip(frame_list_flatten, sem_mask_list_flatten)
		# ]

		gt_mask = [sem_segm_tool(frames[n_steps+1], [prompt]) for frames in frame_list]

		recon_frame_list = []
		for i, frames in enumerate(frame_list):
			inpainting_tool.reset()
			_frames = frame_list_flatten[i*n_steps:(i+1)*n_steps]
			_masks = sem_mask_list_flatten[i*n_steps:(i+1)*n_steps]
			for j, (frame, mask) in enumerate(zip(_frames, _masks)):
				masked_frame: PIL.Image = PIL.Image.composite(frame, PIL.Image.new("RGB", frame.size), mask)
				recon_frame: PIL.Image = inpainting_tool(masked_frame, mask)

			recon_frame_list.append(recon_frame)

		inputs = refiner_processor(images=recon_frame_list, text=len(recon_frame_list)*[prompt], return_tensors="pt").to(device)
		outputs: RefinerDecoderOutput = refiner(**inputs)
		logits, values = outputs.logits.permute(0, 2, 3, 1), outputs.values
		dist = torch.distributions.Categorical(logits=logits)
		action = dist.sample()

		gt = torch.stack([
			torch.tensor(np.array(mask.resize((352, 352))))
			for mask in gt_mask
		]).to(device)
		rewards = (action == gt).float()

		# advantage
		advantages = rewards - values.detach()

		policy_loss = -(dist.log_prob(action) * advantages).mean()
		value_loss = F.mse_loss(values, rewards)

		entropy = dist.entropy().mean()

		loss = (
			policy_loss
			+ 0.5 * value_loss
			- 0.01 * entropy
		)

		optimizer.zero_grad()
		loss.backward()
		optimizer.step()

		print(f'\rEpoch: {epoch}, loss: {loss.item():.4f}', end="")

		# break
	print()
	break

Epoch: 0, loss: 0.00360


In [12]:
logits.shape, action.shape, values.shape

(torch.Size([2, 352, 352, 2]),
 torch.Size([2, 352, 352]),
 torch.Size([2, 352, 352]))

In [28]:
outputs.logits.shape

torch.Size([2, 2, 352, 352])